# Retail.xlsx: audit-first pipeline for BI

Этот notebook задуман как рабочий журнал разбора `Retail.xlsx`: сначала идёт первичный осмотр данных и проверка гипотез, затем фиксация наблюдений, и только после этого запуск внешнего pipeline из `src/`. Финальная трансформационная логика живёт во внешних модулях, а здесь важен именно ход исследования и связь между наблюдениями и решениями.


## 1. Инициализация

Что делаем: загружаем конфигурацию проекта, читаем исходный Excel и подключаем функции аудита.


In [13]:
# File: c:\Git\MFTI\DataVizualize\Финальный проект_ИАВД-11\Финальный проект_источники\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
# ruff: noqa: E402
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    for candidate in [project_root, *project_root.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            project_root = candidate
            break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.classification import classify_line_type
from src.config import PipelineConfig
from src.io_utils import load_retail_data
from src.normalization import apply_normalization
from src.pipeline import run_pipeline
from src.profiling import (
    build_basic_profile,
    build_country_mapping_table,
    build_extreme_rows,
    build_last_month_summary,
    build_line_type_summary,
    build_missingness_profile,
    build_stock_description_issues,
    build_text_noise_summary,
    find_anonymous_transactions,
    find_bad_debt_candidates,
    find_business_duplicates,
    find_exact_duplicates,
    find_missing_description_rows,
    find_return_candidates,
    find_service_code_rows,
    find_zero_price_rows,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)

cfg = PipelineConfig()
raw_df, source_path = load_retail_data(cfg)
normalized_df = apply_normalization(raw_df, cfg)
audited_df = classify_line_type(normalized_df, cfg)
source_path

WindowsPath('C:/Git/MFTI/DataVizualize/final_project/retail_bi_pipeline/data/raw/Retail.xlsx')

## 2. Базовое профилирование

Сначала смотрим на размер набора, состав полей, типы, период и базовую полноту.

На этом шаге важно не придумывать правила очистки заранее, а понять, где данные выглядят устойчиво, а где уже видны подозрительные зоны.


In [14]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
print("source:", source_path)
print("shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes.to_frame("dtype"))
display(build_basic_profile(raw_df))
display(build_missingness_profile(raw_df))

source: C:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\data\raw\Retail.xlsx
shape: (947217, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085.0,United Kingdom,organic,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,organic,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085.0,United Kingdom,organic,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,organic,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,organic,False


,dtype
Invoice,string[python]
StockCode,string[python]
Description,string[python]
Quantity,Int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,string[python]
Channel,string[python]
rnd,boolean


,metric,value
0,row_count,947217
1,column_count,10
2,invoice_unique,48753
3,customer_unique_non_null,5695
4,country_unique,43
5,date_min,2009-12-01 00:00:00
6,date_max,2011-10-27 00:00:00
7,invoice_time_unique,1


,column_name,missing_count,missing_pct
0,Channel,214018,0.225944
1,Customer ID,214018,0.225944
2,Description,4287,0.004526
3,Country,0,0.000000
4,Invoice,0,0.000000
5,InvoiceDate,0,0.000000
6,Price,0,0.000000
7,Quantity,0,0.000000
8,StockCode,0,0.000000
9,rnd,0,0.000000


### Наблюдения и рабочий вывод

- Набор крупный: `947 217` строк, `10` полей и `48 753` уникальных инвойсов. Даже локальные ошибки здесь будут заметны в агрегатах.
- Пропуски распределены не случайно: `Customer ID` и `Channel` одновременно отсутствуют в `214 018` строках (`22.6%`), а `Description` отсутствует в `4 287` строках.
- Это пока не готовое правило очистки, а сигнал для следующих проверок: отдельно стоит разобрать анонимные транзакции, описания и справочники.


## 3. Дубликаты

Сначала разбираем повторы строк. Это тот случай, где похожие записи ещё не обязательно означают ошибку, поэтому здесь важно сразу договориться о терминах.

Под **полным дубликатом** в этом notebook понимается буквальное совпадение всей строки исходного Excel.
Под **дубликатом по бизнес-ключу** понимается совпадение набора полей `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`, `Channel`.

Это важное различие: полный дубль почти всегда похож на технический повтор одной и той же записи, а совпадение по бизнес-ключу может означать как лишнюю копию, так и две реально проведённые одинаковые операции.


In [15]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
exact_dups = find_exact_duplicates(raw_df)
business_dups = find_business_duplicates(raw_df, cfg)
print("exact duplicates:", len(exact_dups))
print("business duplicates:", len(business_dups))
display(exact_dups.head())
display(business_dups.head())

exact duplicates: 59730
business duplicates: 63858


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,sales,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,sales,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01,3.75,16329.0,United Kingdom,sales,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01,0.65,16329.0,United Kingdom,sales,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01,0.85,16329.0,United Kingdom,sales,False


### Наблюдения и рабочий вывод

- Полных дублей найдено `59 730`, дублей по бизнес-ключу `63 858`.
- Полный дубль здесь выглядит самым сильным кандидатом на технический шум: совпадает не часть атрибутов, а вся строка целиком.
- С бизнес-дублями ситуация аккуратнее. Совпадение по бизнес-ключу не доказывает ошибку, а лишь помечает строки как подозрительно одинаковые с точки зрения операции.
- Поэтому решение разделено: полные дубли не тянем в факт как самостоятельные строки, а бизнес-дубли не удаляем автоматически и сохраняем как отдельный сигнал `is_business_duplicate` для QA и последующих BI-проверок.


## 4. Возвраты и сторно

Сравниваем два простых признака возврата: `Invoice` на `C` и отрицательное `Quantity`.

Если они не совпадут, придётся принимать решение уже по данным, а не по удобному правилу.


In [16]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
returns_df = find_return_candidates(raw_df)
invoice_c = raw_df["Invoice"].astype("string").fillna("").str.upper().str.startswith("C")
qty_negative = raw_df["Quantity"] < 0
print("invoice startswith C:", int(invoice_c.sum()))
print("quantity < 0:", int(qty_negative.sum()))
print("negative qty but not C:", int((qty_negative & ~invoice_c).sum()))
print("C invoice but positive qty:", int((invoice_c & (raw_df["Quantity"] > 0)).sum()))
display(returns_df.head())

invoice startswith C: 17980
quantity < 0: 21234
negative qty but not C: 3255
C invoice but positive qty: 1


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01,2.95,16321.0,Australia,sales,False
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01,1.65,16321.0,Australia,sales,False
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01,4.25,16321.0,Australia,sales,False
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01,2.10,16321.0,Australia,sales,False
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01,2.95,16321.0,Australia,sales,False


### Наблюдения и рабочий вывод

- `Invoice` на `C` даёт `17 980` строк, отрицательное `Quantity` - `21 234`.
- Есть `3 255` строк с отрицательным количеством без `C` и `1` строка с `C`, но положительным количеством.
- Значит, одного признака недостаточно; разумнее опираться на комбинированную логику возврата.


## 5. Bad debt и бухгалтерские корректировки

Проверяем, не скрывается ли в источнике отдельный слой бухгалтерских корректировок.

Для этого смотрим на сочетание `Invoice` на `A`, `StockCode = B`, отрицательной цены и описаний строк.


In [17]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
bad_debt_df = find_bad_debt_candidates(raw_df)
display(bad_debt_df)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
179403,A506401,B,Adjust bad debt,1,2010-04-29,-53594.36,NaN,United Kingdom,<NA>,True
276274,A516228,B,Adjust bad debt,1,2010-07-19,-44031.79,NaN,United Kingdom,<NA>,True
403472,A528059,B,Adjust bad debt,1,2010-10-20,-38925.87,NaN,United Kingdom,<NA>,True
825443,A563185,B,Adjust bad debt,1,2011-08-12,11062.06,NaN,United Kingdom,<NA>,False
825444,A563186,B,Adjust bad debt,1,2011-08-12,-11062.06,NaN,United Kingdom,<NA>,True
825445,A563187,B,Adjust bad debt,1,2011-08-12,-11062.06,NaN,United Kingdom,<NA>,True


### Наблюдения и рабочий вывод

- В выборке есть строки с `Invoice` на `A`, `StockCode = B` и крупными отрицательными суммами по `Adjust bad debt`.
- По смыслу они больше похожи на бухгалтерские корректировки, чем на продажу или возврат товара.
- Поэтому их лучше выносить в отдельный сервисный тип и не смешивать с товарным контуром.


## 6. Нулевая цена и служебные строки

Смотрим, насколько часто встречаются `Price = 0` и явные нетоварные `StockCode`.

Сначала фиксируем масштаб и примеры, а уже потом решаем, что считать служебными строками.


In [18]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
zero_price_df = find_zero_price_rows(raw_df)
service_df = find_service_code_rows(raw_df, cfg)
print("zero price rows:", len(zero_price_df))
print("service code rows:", len(service_df))
display(zero_price_df.head())
display(service_df[["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(20))

zero price rows: 5837
service code rows: 5173


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
263,489464,21733,85123a mixed,-96,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
283,489463,71477,short,-240,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
284,489467,85123A,21733 mixed,-192,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
470,489521,21646,<NA>,-50,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3114,489655,20683,<NA>,-44,2009-12-01,0.0,NaN,United Kingdom,<NA>,True


,Invoice,StockCode,Description,Quantity,Price
89,489439,POST,POSTAGE,3,18.00
126,489444,POST,POSTAGE,1,141.00
173,489447,POST,POSTAGE,1,130.00
625,489526,POST,POSTAGE,6,18.00
735,C489535,D,Discount,-1,9.00
736,C489535,D,Discount,-1,19.00
927,C489538,POST,POSTAGE,-1,9.58
1244,489557,POST,POSTAGE,4,18.00
2379,489597,DOT,DOTCOM POSTAGE,1,647.19
2539,489600,DOT,DOTCOM POSTAGE,1,55.96


### Наблюдения и рабочий вывод

- Строк с `Price = 0` (`5 837`) и явных служебных `StockCode` (`5 173`) слишком много, чтобы считать их редким шумом.
- По примерам видно, что сюда попадают разные случаи: `POSTAGE`, `Discount`, строки с неполным описанием и другие нетоварные записи.
- Значит, такие строки лучше сначала отделить от обычных продаж, а затем решать, какие из них нужны в BI отдельно.


## 7. Клиенты, канал, описание

Здесь начинается основная часть предобработки: не редкие аномалии, а массовые пропуски и неполные атрибуты.

Нас интересуют три разных случая: нет `Customer ID`, нет `Channel`, нет `Description`. Формально это всё пропуски, но для будущей модели они означают разное, поэтому одного общего правила вроде «заполнить все NA» здесь недостаточно.


In [19]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
anonymous_df = find_anonymous_transactions(raw_df)
missing_desc_df = find_missing_description_rows(raw_df)
print("anonymous rows:", len(anonymous_df))
print("missing description rows:", len(missing_desc_df))
print(
    "channel missing & customer missing:",
    int((raw_df["Channel"].isna() & raw_df["Customer ID"].isna()).sum()),
)
print(
    "channel missing & customer present:",
    int((raw_df["Channel"].isna() & raw_df["Customer ID"].notna()).sum()),
)
display(anonymous_df.head())
display(missing_desc_df.head())

anonymous rows: 214018
missing description rows: 4287
channel missing & customer missing: 214018
channel missing & customer present: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
263,489464,21733,85123a mixed,-96,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
283,489463,71477,short,-240,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
284,489467,85123A,21733 mixed,-192,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
470,489521,21646,<NA>,-50,2009-12-01,0.00,NaN,United Kingdom,<NA>,True
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01,0.55,NaN,United Kingdom,<NA>,False


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Channel,rnd
470,489521,21646,<NA>,-50,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3114,489655,20683,<NA>,-44,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3161,489659,21350,<NA>,230,2009-12-01,0.0,NaN,United Kingdom,<NA>,True
3731,489781,84292,<NA>,17,2009-12-02,0.0,NaN,United Kingdom,<NA>,True
4296,489806,18010,<NA>,-770,2009-12-02,0.0,NaN,United Kingdom,<NA>,True


### Наблюдения и рабочий вывод

- `214 018` строк идут без `Customer ID`, и в этих же строках отсутствует `Channel`. Это не похоже на случайный шум по одной колонке; скорее, источник системно не передал клиентский контур для части операций.
- Строк без `Description` заметно меньше (`4 287`), но для товарной аналитики они чувствительны: сама транзакция существует, а товарный атрибут уже неполный.
- Отдельно важно, что в ключевых полях сырого файла почти нет именно пустых строк `''`. Основной случай здесь не blank-строки, а настоящие `NA`, поэтому и решения принимаются именно для пропусков.
- Рабочие решения получаются разными по смыслу:
  - `Customer ID` не восстанавливаем искусственно и не выбрасываем такие продажи, а переводим в отдельную метку `ANONYMOUS`.
  - `Channel` не угадываем по косвенным признакам, а явно фиксируем как `UNKNOWN` / `unknown`.
  - `Description` не заполняем выдуманным текстом: пропуск сохраняется как ограничение, а строки без описания или со служебными подписями вроде `DAMAGED`, `FOUND`, `CHECK` дополнительно помечаются через `is_description_placeholder`.
- Иными словами, здесь предобработка не «лечит» данные до идеального вида, а делает пропуски явными и управляемыми для BI.


## 8. Справочник товаров, текстовый шум и география

Проверяем, насколько устойчивы товарные и географические атрибуты: описания, коды, пробелы, регистр, написание стран.

Здесь легко переоценить `Description`, хотя на практике оно может оказаться нестабильным атрибутом.


In [20]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_stock_description_issues(audited_df).head(20))
display(build_text_noise_summary(raw_df))
display(build_country_mapping_table(audited_df))

,issue_type,entity,variant_count
1132,description_to_many_stock_codes,<NA>,2417
1041,description_to_many_stock_codes,?,88
1064,description_to_many_stock_codes,DAMAGED,85
1065,description_to_many_stock_codes,DAMAGES,70
1091,description_to_many_stock_codes,MISSING,28
1077,description_to_many_stock_codes,FOUND,27
1055,description_to_many_stock_codes,CHECK,22
1116,description_to_many_stock_codes,SOLD AS SET ON DOTCOM,20
1045,description_to_many_stock_codes,ADJUSTMENT,12
1071,description_to_many_stock_codes,DOTCOM,11


,metric,value
0,description_trim_changed,184479
1,description_multiple_spaces,48058
2,stock_code_not_upper,3124


,country_raw,country_norm
0,Australia,Australia
1,Austria,Austria
2,Bahrain,Bahrain
3,Belgium,Belgium
4,Bermuda,Bermuda
...,...,...
38,United Arab Emirates,United Arab Emirates
39,United Kingdom,United Kingdom
40,USA,United States
41,Unspecified,Unknown


### Наблюдения и рабочий вывод

- В исходнике нет массовых пустых строк по ключевым полям, зато много текстового шума: `184 479` описаний меняются после trim, `48 058` содержат лишние пробелы, `3 124` `StockCode` записаны не в верхнем регистре.
- Кроме косметического шума есть структурная проблема: одно описание может относиться к нескольким кодам, поэтому `Description` не выглядит надёжным ключом само по себе.
- Решения здесь намеренно простые и воспроизводимые: схлопываем повторяющиеся пробелы, убираем пробелы по краям, нормализуем служебные текстовые поля, а страны приводим к каноническим названиям вроде `EIRE -> Ireland`, `RSA -> South Africa`, `USA -> United States`.
- Поэтому продуктовый контур лучше строить вокруг нормализованного `StockCode`, а `Description` использовать как вспомогательный текстовый атрибут, который тоже пришлось предварительно привести в порядок.


## 9. Экстремумы и неполный последний месяц

Смотрим на экстремальные `Quantity` и `Price`, а также на полноту последнего месяца.

Здесь особенно важно не принять редкую, но реальную операцию за ошибку только из-за масштаба отклонения.


In [21]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_extreme_rows(raw_df, top_n=10))
display(build_last_month_summary(raw_df))

,extreme_metric,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country,Channel
0,quantity,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom,organic
1,quantity,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346.0,United Kingdom,organic
2,quantity,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0,Denmark,sales
3,quantity,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0,Denmark,sales
4,quantity,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0,Denmark,sales
5,quantity,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0,Denmark,sales
6,quantity,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0,Denmark,sales
7,quantity,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,NaN,United Kingdom,<NA>
8,quantity,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0,United Kingdom,performance
9,quantity,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,17940.0,United Kingdom,performance


,max_invoice_date,last_month_start,calendar_month_end,is_last_month_complete
0,2011-10-27,2011-10-01,2011-10-31,False


### Наблюдения и рабочий вывод

- Экстремальные значения в данных есть, но по примерам они не выглядят автоматически ошибочными: среди них встречаются сторно и крупные партии.
- Данные заканчиваются на `2011-10-27`, поэтому последний месяц в выборке неполный.
- Безопаснее сохранять экстремумы в QA и отдельно помечать неполный месяц, чем агрессивно чистить такие строки.


## 10. Итоговая классификация `line_type`

После отдельных проверок смотрим, во что складывается итоговая раскладка `line_type`.

Это уже не стартовая гипотеза, а сводный результат всех предыдущих наблюдений и решений.


In [22]:
# File: c:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\retail_audit_and_pipeline.ipynb
display(build_line_type_summary(audited_df))
display(
    audited_df[["Invoice", "StockCode", "Description", "Quantity", "Price", "line_type"]].head(20)
)

,line_type,row_count,line_amount_sum,quantity_sum
6,sale,919389,1.789656e+07,10258516
5,return,20114,-5.262523e+05,-916576
7,shipping,3233,3.599188e+05,11577
9,unknown,2541,0.000000e+00,227162
4,manual_adjustment,1512,-8.112814e+04,31
2,discount,163,-1.295774e+04,-2858
1,commission_fee,143,-2.469886e+05,-83
3,gift_voucher,99,1.661610e+03,261
8,test,17,2.035000e+02,57
0,bad_debt,6,-1.476141e+05,6


,Invoice,StockCode,Description,Quantity,Price,line_type
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,sale
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,sale
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,sale
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,sale
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,sale
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,1.65,sale
6,489434,21871,SAVE THE PLANET MUG,24,1.25,sale
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,sale
8,489435,22350,CAT BOWL,12,2.55,sale
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,3.75,sale


### Наблюдения и рабочий вывод

- После предыдущих шагов основная масса строк складывается в товарный контур: `sale` содержит `919 389` строк, `return` ещё `20 114`.
- При этом остаются отдельные сегменты `shipping`, `manual_adjustment` и `unknown`, то есть разные типы операций уже не смешаны в одном факте.
- На этом этапе классификация выглядит как итог разбора данных, а не как схема, заданная заранее.


## 11. Запуск production-like pipeline

Ниже notebook вызывает внешний pipeline из `src.pipeline`. Важно, что теперь pipeline не только сохраняет parquet/csv-витрины, но и собирает один итоговый Excel-workbook для загрузки в Yandex DataLens. В этом блоке фиксируем абсолютный путь к созданному workbook, чтобы он не терялся среди остальных артефактов.


In [23]:
from pathlib import Path
import json
from types import SimpleNamespace

summary_path_obj = cfg.reports_dir / "pipeline_run_summary.json"

try:
    result = run_pipeline(cfg)
except PermissionError:
    summary_payload = json.loads(summary_path_obj.read_text(encoding="utf-8"))
    workbook_path = summary_payload.get("datalens_workbook_path")
    if workbook_path and not Path(workbook_path).is_absolute():
        workbook_path = str((cfg.project_root / workbook_path).resolve())
    result = SimpleNamespace(
        source_path=summary_payload["source_path"],
        datalens_workbook_path=workbook_path,
        summary_path=str(summary_path_obj.relative_to(cfg.project_root)),
        exported_processed_files=summary_payload.get("processed_files", []),
        exported_qa_files=summary_payload.get("qa_files", []),
    )
    print("Pipeline rerun skipped because the workbook file is locked; using existing exported artifacts.")

datalens_workbook_path = Path(result.datalens_workbook_path) if result.datalens_workbook_path else None
print(f"Source: {result.source_path}")
print(f"DataLens workbook: {datalens_workbook_path}")
print(f"Summary: {result.summary_path}")
result


Source: C:\Git\MFTI\DataVizualize\final_project\retail_bi_pipeline\data\raw\Retail.xlsx
DataLens workbook: data\processed\retail_datalens_export.xlsx
Summary: reports\pipeline_run_summary.json


PipelineRunResult(source_path='C:\\Git\\MFTI\\DataVizualize\\final_project\\retail_bi_pipeline\\data\\raw\\Retail.xlsx', exported_processed_files=['data\\processed\\fact_sales_lines.parquet', 'data\\processed\\fact_sales_lines.csv', 'data\\processed\\fact_return_lines.parquet', 'data\\processed\\fact_return_lines.csv', 'data\\processed\\fact_service_lines.parquet', 'data\\processed\\fact_service_lines.csv', 'data\\processed\\dim_product.parquet', 'data\\processed\\dim_product.csv', 'data\\processed\\dim_customer.parquet', 'data\\processed\\dim_customer.csv', 'data\\processed\\dim_date.parquet', 'data\\processed\\dim_date.csv', 'data\\processed\\dim_country.parquet', 'data\\processed\\dim_country.csv'], exported_qa_files=['data\\qa\\raw_profile.parquet', 'data\\qa\\raw_profile.csv', 'data\\qa\\raw_missingness.parquet', 'data\\qa\\raw_missingness.csv', 'data\\qa\\duplicate_rows.parquet', 'data\\qa\\duplicate_rows.csv', 'data\\qa\\business_duplicate_rows.parquet', 'data\\qa\\business_dupl

## 12. Структура финального Excel для DataLens

Итоговый workbook специально собран как **один файл с несколькими листами**, а не как один большой «очищенный Excel». Для импорта в DataLens это удобный компромисс: загружать нужно один файл, но внутри остаётся нормальная модель с отдельными фактами и измерениями.

Ниже код показывает фактическую структуру workbook, количество строк и столбцов по листам. Табличная сводка полезна для проверки, но ключевые договорённости лучше зафиксировать ещё и текстом, чтобы смысл не терялся за обрезанными колонками и служебными названиями.


In [24]:
from openpyxl import load_workbook

processed_dir = cfg.processed_dir
qa_dir = cfg.qa_dir

fact_sales = pd.read_parquet(processed_dir / "fact_sales_lines.parquet")
fact_returns = pd.read_parquet(processed_dir / "fact_return_lines.parquet")
fact_service = pd.read_parquet(processed_dir / "fact_service_lines.parquet")
dim_product = pd.read_parquet(processed_dir / "dim_product.parquet")
dim_customer = pd.read_parquet(processed_dir / "dim_customer.parquet")
dim_date = pd.read_parquet(processed_dir / "dim_date.parquet")
dim_country = pd.read_parquet(processed_dir / "dim_country.parquet")
line_type_summary = pd.read_parquet(qa_dir / "line_type_summary.parquet")

workbook = load_workbook(datalens_workbook_path, read_only=True)
sheet_summary = []
for sheet_name, frame in {
    "fact_sales_lines": fact_sales,
    "fact_return_lines": fact_returns,
    "fact_service_lines": fact_service,
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_date": dim_date,
    "dim_country": dim_country,
}.items():
    bool_cols = [c for c in frame.columns if str(frame[c].dtype) in {"bool", "boolean"}]
    sheet_summary.append({
        "sheet_name": sheet_name,
        "row_count": len(frame),
        "column_count": len(frame.columns),
        "bool_flags": ", ".join(bool_cols) if bool_cols else "-",
    })

flag_description = pd.DataFrame([
    {"flag_name": "is_duplicate", "where_used": "fact_*", "why_kept": "Полный дубль в финальных фактах уже снят, поэтому здесь флаг нужен как явный аудит-след и контроль целостности схемы; в итоговых фактах он должен быть False."},
    {"flag_name": "is_business_duplicate", "where_used": "fact_*", "why_kept": "Позволяет быстро выделять подозрительные повторные операции по бизнес-ключу и проверять, не искажают ли они метрики."},
    {"flag_name": "is_service_line", "where_used": "fact_* / dim_product", "why_kept": "Помогает отделять нетоварные строки и сервисные коды от товарного контура без повторной логики классификации в BI."},
    {"flag_name": "is_anonymous_customer", "where_used": "fact_* / dim_customer", "why_kept": "Даёт быстрый фильтр по анонимным транзакциям и позволяет анализировать их как отдельный сегмент без сложных условий."},
    {"flag_name": "is_month_end", "where_used": "dim_date", "why_kept": "Полезен для календарных срезов, сверок закрытия месяца и аккуратных периодических агрегатов."},
    {"flag_name": "is_last_incomplete_month", "where_used": "dim_date", "why_kept": "Нужен, чтобы в графиках не сравнивать неполный последний месяц с полными месяцами как равный период."},
])

display(pd.DataFrame(sheet_summary))
display(flag_description)
display(fact_sales.head())
display(fact_returns.head())
display(fact_service.head())
display(dim_product.head())
display(line_type_summary)
print("Workbook sheets:", workbook.sheetnames)


,sheet_name,row_count,column_count,bool_flags
0,fact_sales_lines,889550,20,"is_duplicate, is_business_duplicate, is_servic..."
1,fact_return_lines,19713,20,"is_duplicate, is_business_duplicate, is_servic..."
2,fact_service_lines,7488,20,"is_duplicate, is_business_duplicate, is_servic..."
3,dim_product,5084,7,is_service_code
4,dim_customer,5696,7,is_anonymous_customer
5,dim_date,696,9,"is_month_end, is_last_incomplete_month"
6,dim_country,43,1,-


,flag_name,where_used,why_kept
0,is_duplicate,fact_*,"Полный дубль в финальных фактах уже снят, поэт..."
1,is_business_duplicate,fact_*,Позволяет быстро выделять подозрительные повто...
2,is_service_line,fact_* / dim_product,Помогает отделять нетоварные строки и сервисны...
3,is_anonymous_customer,fact_* / dim_customer,Даёт быстрый фильтр по анонимным транзакциям и...
4,is_month_end,dim_date,"Полезен для календарных срезов, сверок закрыти..."
5,is_last_incomplete_month,dim_date,"Нужен, чтобы в графиках не сравнивать неполный..."


,raw_record_id,invoice_id,invoice_date,customer_id,channel,country_raw,country_norm,stock_code_raw,stock_code_norm,description_raw,description_norm,product_name_canonical,quantity,unit_price,line_amount,line_type,is_duplicate,is_business_duplicate,is_service_line,is_anonymous_customer
0,1,489434,2009-12-01,13085,organic,United Kingdom,United Kingdom,85048,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,15CM CHRISTMAS GLASS BALL 20 LIGHTS,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4,sale,False,False,False,False
1,2,489434,2009-12-01,13085,organic,United Kingdom,United Kingdom,79323P,79323P,PINK CHERRY LIGHTS,PINK CHERRY LIGHTS,PINK CHERRY LIGHTS,12,6.75,81.0,sale,False,False,False,False
2,3,489434,2009-12-01,13085,organic,United Kingdom,United Kingdom,79323W,79323W,WHITE CHERRY LIGHTS,WHITE CHERRY LIGHTS,WHITE CHERRY LIGHTS,12,6.75,81.0,sale,False,False,False,False
3,4,489434,2009-12-01,13085,organic,United Kingdom,United Kingdom,22041,22041,"RECORD FRAME 7"" SINGLE SIZE","RECORD FRAME 7"" SINGLE SIZE","RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8,sale,False,False,False,False
4,5,489434,2009-12-01,13085,organic,United Kingdom,United Kingdom,21232,21232,STRAWBERRY CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0,sale,False,False,False,False


,raw_record_id,invoice_id,invoice_date,customer_id,channel,country_raw,country_norm,stock_code_raw,stock_code_norm,description_raw,description_norm,product_name_canonical,quantity,unit_price,line_amount,line_type,is_duplicate,is_business_duplicate,is_service_line,is_anonymous_customer
0,179,C489449,2009-12-01,16321,sales,Australia,Australia,22087,22087,PAPER BUNTING WHITE LACE,PAPER BUNTING WHITE LACE,PAPER BUNTING WHITE LACE,-12,2.95,-35.4,return,False,False,False,False
1,180,C489449,2009-12-01,16321,sales,Australia,Australia,85206A,85206A,CREAM FELT EASTER EGG BASKET,CREAM FELT EASTER EGG BASKET,CREAM FELT EASTER EGG BASKET,-6,1.65,-9.9,return,False,False,False,False
2,181,C489449,2009-12-01,16321,sales,Australia,Australia,21895,21895,POTTING SHED SOW 'N' GROW SET,POTTING SHED SOW 'N' GROW SET,POTTING SHED SOW 'N' GROW SET,-4,4.25,-17.0,return,False,False,False,False
3,182,C489449,2009-12-01,16321,sales,Australia,Australia,21896,21896,POTTING SHED TWINE,POTTING SHED TWINE,POTTING SHED TWINE,-6,2.10,-12.6,return,False,False,False,False
4,183,C489449,2009-12-01,16321,sales,Australia,Australia,22083,22083,PAPER CHAIN KIT RETRO SPOT,PAPER CHAIN KIT RETRO SPOT,PAPER CHAIN KIT RETROSPOT,-12,2.95,-35.4,return,False,False,False,False


,raw_record_id,invoice_id,invoice_date,customer_id,channel,country_raw,country_norm,stock_code_raw,stock_code_norm,description_raw,description_norm,product_name_canonical,quantity,unit_price,line_amount,line_type,is_duplicate,is_business_duplicate,is_service_line,is_anonymous_customer
0,90,489439,2009-12-01,12682,organic,France,France,POST,POST,POSTAGE,POSTAGE,POSTAGE,3,18.0,54.0,shipping,False,False,True,False
1,127,489444,2009-12-01,12636,sales,USA,United States,POST,POST,POSTAGE,POSTAGE,POSTAGE,1,141.0,141.0,shipping,False,False,True,False
2,174,489447,2009-12-01,12362,organic,Belgium,Belgium,POST,POST,POSTAGE,POSTAGE,POSTAGE,1,130.0,130.0,shipping,False,False,True,False
3,626,489526,2009-12-01,12533,sales,Germany,Germany,POST,POST,POSTAGE,POSTAGE,POSTAGE,6,18.0,108.0,shipping,False,False,True,False
4,736,C489535,2009-12-01,15299,sales,United Kingdom,United Kingdom,D,D,Discount,DISCOUNT,DISCOUNT,-1,9.0,-9.0,discount,False,False,True,False


,stock_code_norm,description_variant_count,source_row_count,first_seen_date,last_seen_date,product_name_canonical,is_service_code
0,10002,1,400,2009-12-01,2011-04-28,INFLATABLE POLITICAL GLOBE,False
1,10002R,1,3,2009-12-02,2010-01-25,ROBOT PENCIL SHARPNER,False
2,10080,1,24,2009-12-02,2011-10-25,GROOVY CACTUS INFLATABLE,False
3,10109,1,2,2009-12-03,2010-02-04,BENDY COLOUR PENCILS,False
4,10120,2,70,2009-12-01,2011-10-23,DOGGY RUBBER,False


,line_type,row_count,line_amount_sum,quantity_sum
0,sale,919389,1.789656e+07,10258516
1,return,20114,-5.262523e+05,-916576
2,shipping,3233,3.599188e+05,11577
3,unknown,2541,0.000000e+00,227162
4,manual_adjustment,1512,-8.112814e+04,31
5,discount,163,-1.295774e+04,-2858
6,commission_fee,143,-2.469886e+05,-83
7,gift_voucher,99,1.661610e+03,261
8,test,17,2.035000e+02,57
9,bad_debt,6,-1.476141e+05,6


Workbook sheets: ['fact_sales_lines', 'fact_return_lines', 'fact_service_lines', 'dim_product', 'dim_customer', 'dim_date', 'dim_country']


### Как читать итоговый workbook

`fact_sales_lines`, `fact_return_lines`, `fact_service_lines` разделены не ради красоты. Это три разных контура поведения данных: обычные продажи, возвраты и нетоварные сервисные / бухгалтерские строки. Если склеить их обратно в один лист, то почти любая метрика в DataLens начнёт требовать дополнительных условий и быстро станет хрупкой.

`dim_product`, `dim_customer`, `dim_date`, `dim_country` нужны по той же причине: измерения хранятся отдельно, чтобы в датасете можно было спокойно джойнить атрибуты и не раздувать факт повторяющимися полями.

### Что означают bool-флаги в workbook

- `is_duplicate` — строка участвовала в полном дубле, то есть целиком совпадала с другой строкой исходного Excel. В итоговых фактах этот флаг почти всегда не нужен как активная аналитическая ось, потому что полные дубли снимаются ещё до финального слоя; он сохранён скорее для прозрачности и QA.
- `is_business_duplicate` — строка совпадает с другой по бизнес-ключу (`Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`, `Channel`). Это не «ошибка по определению», а сигнал проверить подозрительно одинаковые операции.
- `is_service_line` / `is_service_code` — строка или код относятся к нетоварному контуру: доставка, скидка, комиссия, ручная корректировка, тестовая или иная служебная операция. Такой флаг помогает быстро отделять товарную аналитику от сервисной.
- `is_anonymous_customer` — операция прошла без клиентского идентификатора. Это полезный срез сам по себе: такие строки не удалены, но их удобно анализировать отдельно.
- `is_month_end` — дата является последним календарным днём месяца.
- `is_last_incomplete_month` — дата попадает в последний месяц источника, который обрывается раньше календарного конца. Такой флаг нужен, чтобы в BI не смешивать полный и неполный месяц в одной трендовой логике.

Практический смысл этих флагов простой: они позволяют в DataLens фильтровать спорные или специальные сегменты без повторной логики на уровне графиков.


## 13. Финальные выводы для BI

После этого разбора итоговая логика уже не выглядит как «просто почистили Excel», а сводится к нескольким явным решениям.

- Исходный файл не годится как единая факт-таблица: в нём смешаны продажи, возвраты, сервисные строки и бухгалтерские корректировки.
- Полные дубли считаем техническим шумом и не тянем в финальный факт, а совпадения по бизнес-ключу не удаляем вслепую, а сохраняем как проверочный сигнал.
- Пропуски не маскируем общей заливкой: отсутствующий клиент уходит в `ANONYMOUS`, отсутствующий канал — в `UNKNOWN`, а пропавшее или техническое описание не выдумывается, а явно помечается как ограничение.
- Текстовая очистка здесь не декоративная: без trim, нормализации регистра и приведения стран группировки в BI были бы шумными и плохо воспроизводимыми.
- Для DataLens удобнее не россыпь `csv`, а один workbook, где листы уже разложены по будущей логике датасета и графиков.
